Clustering and pi0 Candidates in the June Contiguous Calorimeter Setup

In this notebook you will work with the June testbeam run `1781576899`. The goal is to understand how calorimeter cells become clusters, and how two electromagnetic clusters can be used as pi0 -> gamma gamma candidates.

This version contains questions and TODO cells as it is designed for students:) Try to reason from the detector layout and the event displays before comparing to the prepared pi0 monitor output.


## Logbook and Data

Use the June BL4S 2026 logbook while working through this notebook:

- [June logbook](https://codimd.web.cern.ch/yB8Ihfl4QGGRWXDTChaglQ#20260613)

Main run for this exercise: `1781576899`.

The notebook uses two files from the June testbeam area:

- Raw/reconstructed event file: `/Users/berare/BL4S/TDAQ_stateofArt/1781576899.root`
- Prepared pi0 monitor file with clustering vectors: `/Users/berare/BL4S/TDAQ_stateofArt/pi0_monitor_1781576899_vectors.root` **they are part of your convertedToROOT folder, please update accoringly the paths always!!!**

The vector file is useful for checking the result, but the main lesson is to understand how clusters are built from the calorimeter layout.


In [ ]:
import json
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import ROOT
except Exception as exc:
    raise RuntimeError("PyROOT is required. Start Jupyter from an environment where ROOT is set up.") from exc

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["figure.dpi"] = 110

DATA_DIRS = [
    Path("/Users/berare/BL4S/TDAQ_stateofArt"), #don't forget to update your path!
    Path("/Users/berare/BL4S/TDAQ_stateOfArt"),
    Path("."),
    Path(".."),
]
DATA_DIR = next((p for p in DATA_DIRS if (p / "1781576899.root").exists()), DATA_DIRS[0])

RUN = "1781576899"
RAW_FILE = DATA_DIR / f"{RUN}.root"
VECTOR_FILE = DATA_DIR / f"pi0_monitor_{RUN}_vectors.root"
CONFIG_FILE = DATA_DIR / "bl4s2026_juneTestBeam_calorimeters_finalForm.json"

print("Raw file:   ", RAW_FILE, "exists =", RAW_FILE.exists())
print("Vector file:", VECTOR_FILE, "exists =", VECTOR_FILE.exists())
print("Config file:", CONFIG_FILE, "exists =", CONFIG_FILE.exists())


In [ ]:
def open_root_tree(path, tree_name):
    f = ROOT.TFile.Open(str(path))
    if not f or f.IsZombie():
        raise OSError(f"Could not open {path}")
    tree = f.Get(tree_name)
    if not tree:
        raise KeyError(f"Tree {tree_name!r} not found in {path}")
    return f, tree


def hist_to_arrays(hist):
    x = np.array([hist.GetBinCenter(i) for i in range(1, hist.GetNbinsX() + 1)])
    y = np.array([hist.GetBinContent(i) for i in range(1, hist.GetNbinsX() + 1)])
    return x, y


def plot_root_hist(file_path, hist_name, xlabel=None, logy=False):
    f = ROOT.TFile.Open(str(file_path))
    h = f.Get(hist_name)
    if not h:
        f.Close()
        raise KeyError(f"Histogram {hist_name!r} not found")
    x, y = hist_to_arrays(h)
    plt.step(x, y, where="mid")
    if logy:
        plt.yscale("log")
    plt.xlabel(xlabel or hist_name)
    plt.ylabel("entries")
    plt.title(hist_name)
    plt.show()
    print("entries =", int(h.GetEntries()))
    print("mean    =", h.GetMean())
    print("rms     =", h.GetRMS())
    print("peak bin center =", h.GetBinCenter(h.GetMaximumBin()))
    f.Close()


> **Warning: `RECOdata` is already calibrated**
>
> In this notebook the clustering starts from the `RECOdata` tree, not directly from raw QDC counts.
>
> - `RAWdata` contains raw detector readout branches such as `QDC0_ch6`.
> - `RECOdata` contains reconstructed calorimeter amplitudes such as `Cal8_amplitude`.
> - For the June testbeam file used here, the pedestal subtraction and QDC-to-energy conversion have already been applied before the values appear in `RECOdata`.
>
> Therefore, the clustering exercise does **not** ask you to redo pedestal subtraction or build the QDC calibration. Those calibration files live under `QDCConfig/` and are part of how `RECOdata` was produced. In this notebook you should focus on how calibrated cell energies are grouped into clusters and then into two-photon pi0 candidates.


## Part 1 - Open the Files

The raw/reconstructed file contains one row per event. The vector monitor file contains already-produced clustering and pi0 quantities.

**Question 1:** Which file should you use if you want to build clusters yourself from calorimeter energies? Which file should you use as a checkpoint?


In [ ]:
# TODO: open the RECOdata tree from RAW_FILE using open_root_tree.
# Then print the number of events.

# raw_f, reco_tree = open_root_tree(RAW_FILE, "RECOdata")
# print("RECOdata entries:", reco_tree.GetEntries())
# raw_f.Close()


## Part 2 - The Calorimeter Layout

Clustering is only meaningful after you know which detectors are neighbors. In this June setup the calorimeters are arranged in a matrix. Some matrix positions are empty; those empty positions are gaps, not active cells.

Two cells are treated as neighbors if they share an edge: up, down, left, or right. Diagonal touching is not enough for this exercise.

**Question 2:** Look at the matrix. Which calorimeters are direct neighbors of `Cal4`? Which are direct neighbors of `Cal18`?


In [ ]:
with open(CONFIG_FILE) as f:
    config = json.load(f)

LAYOUT = config["BL4S_CalorimeterLayout"]["matrix"]
CAL_DETECTORS = [cell for row in LAYOUT for cell in row if cell is not None]

# Approximate block pitch used for a simple geometry model.
# The exact number is less important here than being consistent when comparing events.
CELL_SIZE_CM = 10.0
TARGET_TO_CAL_CM = 200.0

positions = {}
for row, line in enumerate(LAYOUT):
    for col, det in enumerate(line):
        if det is None:
            continue
        x = (col - 2) * CELL_SIZE_CM
        y = (1.5 - row) * CELL_SIZE_CM
        positions[det] = {"row": row, "column": col, "x_cm": x, "y_cm": y}

print("June calorimeter layout:")
for row in LAYOUT:
    print("  ", row)
print("\nNumber of calorimeter channels in layout:", len(CAL_DETECTORS))


In [ ]:
def plot_layout(values=None, title="June calorimeter layout"):
    fig, ax = plt.subplots(figsize=(8, 5))
    for row, line in enumerate(LAYOUT):
        for col, det in enumerate(line):
            x = col
            y = -row
            if det is None:
                ax.add_patch(plt.Rectangle((x - 0.45, y - 0.45), 0.9, 0.9, fill=False, linestyle="--", color="0.75"))
                ax.text(x, y, "gap", ha="center", va="center", color="0.55", fontsize=9)
                continue
            val = None if values is None else values.get(det, 0.0)
            color = "white" if val is None else plt.cm.viridis(min(1.0, max(0.0, val / max(max(values.values()), 1e-9))))
            ax.add_patch(plt.Rectangle((x - 0.45, y - 0.45), 0.9, 0.9, facecolor=color, edgecolor="black"))
            label = det if val is None else f"{det}\n{val:.0f}"
            ax.text(x, y, label, ha="center", va="center", fontsize=9)
    ax.set_aspect("equal")
    ax.set_xticks(range(len(LAYOUT[0])))
    ax.set_yticks([-r for r in range(len(LAYOUT))])
    ax.set_xlabel("layout column")
    ax.set_ylabel("layout row")
    ax.set_title(title)
    plt.show()

plot_layout()


## Part 3 - Inspect Prepared Clustering Histograms

Before writing your own clustering code, inspect the prepared monitor output. This tells you what kind of cluster multiplicities and cluster energies to expect.

**Question 3:** Is most of the run empty, one-cluster, or multi-cluster? Why do pi0 candidates need at least two clusters?


In [ ]:
def open_root_tree(path, tree_name):
    f = ROOT.TFile.Open(str(path))
    if not f or f.IsZombie():
        raise OSError(f"Could not open {path}")
    tree = f.Get(tree_name)
    if not tree:
        raise KeyError(f"Tree {tree_name!r} not found in {path}")
    return f, tree


def hist_to_arrays(hist):
    x = np.array([hist.GetBinCenter(i) for i in range(1, hist.GetNbinsX() + 1)])
    y = np.array([hist.GetBinContent(i) for i in range(1, hist.GetNbinsX() + 1)])
    return x, y


def plot_root_hist(file_path, hist_name, xlabel=None, logy=False):
    f = ROOT.TFile.Open(str(file_path))
    h = f.Get(hist_name)
    if not h:
        f.Close()
        raise KeyError(f"Histogram {hist_name!r} not found")
    x, y = hist_to_arrays(h)
    plt.step(x, y, where="mid")
    if logy:
        plt.yscale("log")
    plt.xlabel(xlabel or hist_name)
    plt.ylabel("entries")
    plt.title(hist_name)
    plt.show()
    print("entries =", int(h.GetEntries()))
    print("mean    =", h.GetMean())
    print("rms     =", h.GetRMS())
    print("peak bin center =", h.GetBinCenter(h.GetMaximumBin()))
    f.Close()


In [ ]:
plot_root_hist(VECTOR_FILE, "calorimeter_cluster_multiplicity", xlabel="number of clusters per event", logy=True)
plot_root_hist(VECTOR_FILE, "calorimeter_cluster_energy_adc", xlabel="cluster energy / ADC sum", logy=True)
plot_root_hist(VECTOR_FILE, "calorimeter_cluster_size", xlabel="number of cells in cluster", logy=True)


## Part 4 - Build a Cluster by Hand

A simple clustering algorithm has three ideas:

1. **Cell threshold:** ignore very small amplitudes so pedestal/noise cells do not become clusters.
2. **Connectivity:** group active cells that touch by an edge.
3. **Seed threshold:** keep only groups that contain at least one reasonably strong cell.

For each accepted cluster, compute:

- Total energy: sum of all cells in the cluster.
- Cluster size: number of cells in the group.
- Cluster position: energy-weighted average of the cell positions.

**Question 4:** Why is an energy-weighted position better than just using the center of the highest-energy cell?


In [ ]:
def neighbors(cell):
    row, col = cell
    # TODO: return the valid up/down/left/right neighbors.
    # A valid neighbor must stay inside the matrix and must not be a None/gap cell.
    raise NotImplementedError("Fill the neighbor-finding logic")


def read_event_energies(tree, event_index):
    tree.GetEntry(event_index)
    values = {}
    for det in CAL_DETECTORS:
        # TODO: read the reconstructed amplitude branch for this detector.
        # Hint: the branch names look like Cal8_amplitude, Cal14_amplitude, etc.
        values[det] = None
    return values


def cluster_event(energies, cell_threshold=40.0, seed_threshold=80.0):
    # TODO: build clusters from neighboring cells.
    # Suggested steps:
    # 1. Mark every detector with energy > cell_threshold as active.
    # 2. Convert active detector names to layout cells: (row, column).
    # 3. Use a flood-fill or breadth-first search to group adjacent active cells.
    # 4. Reject groups whose largest cell is below seed_threshold.
    # 5. For each remaining group, compute total energy and energy-weighted position.
    # 6. Sort clusters from highest energy to lowest energy.
    raise NotImplementedError("Implement the clustering algorithm")


## Part 5 - Test the Clusterer on One Event

After implementing the functions above, find an event with at least two clusters and draw it.

**Question 5:** Do the two clusters sit in separated regions of the calorimeter? Are they on the same stack or across the central gap?


In [ ]:
raw_f, reco_tree = open_root_tree(RAW_FILE, "RECOdata")

# TODO: once cluster_event works, loop over events until you find one with at least two clusters.
# Try thresholds such as cell_threshold=40 and seed_threshold=80.

# event_index = ...
# energies = read_event_energies(reco_tree, event_index)
# clusters = cluster_event(energies, cell_threshold=40, seed_threshold=80)
# print(event_index, clusters)
# plot_layout(energies, title=f"Run {RUN}, event {event_index}")

raw_f.Close()


## Part 6 - From Two Clusters to pi0 Candidates

For a pi0 -> gamma gamma candidate, treat each electromagnetic cluster as a photon. The two-photon invariant mass is

`m^2 = 2 E1 E2 (1 - cos(theta))`

where `theta` is the opening angle between the two photon directions. In this exercise the photon direction is approximated by a line from the target to the cluster position on the calorimeter face.

**Question 6:** What happens to the invariant mass if the two clusters are very close together? What happens if the opening angle is larger?


In [ ]:
def unit_vector_from_cluster(cluster):
    vec = np.array([cluster["x_cm"], cluster["y_cm"], TARGET_TO_CAL_CM], dtype=float)
    return vec / np.linalg.norm(vec)


def invariant_mass_two_photons(c1, c2):
    # For two photons: m^2 = 2 E1 E2 (1 - cos(theta)).
    # If energies are in MeV, the mass is returned in MeV.
    u1 = unit_vector_from_cluster(c1)
    u2 = unit_vector_from_cluster(c2)
    cos_theta = float(np.clip(np.dot(u1, u2), -1.0, 1.0))
    m2 = 2.0 * c1["energy"] * c2["energy"] * (1.0 - cos_theta)
    return math.sqrt(max(m2, 0.0))


def all_cluster_pairs(clusters):
    pairs = []
    for i in range(len(clusters)):
        for j in range(i + 1, len(clusters)):
            mass = invariant_mass_two_photons(clusters[i], clusters[j])
            sep = math.hypot(clusters[i]["x_cm"] - clusters[j]["x_cm"], clusters[i]["y_cm"] - clusters[j]["y_cm"])
            pairs.append({"i": i, "j": j, "mass": mass, "separation_cm": sep})
    return pairs


In [ ]:
# TODO: after finding an event with at least two clusters, calculate all pair masses.
# pairs = all_cluster_pairs(clusters)
# pairs


## Part 7 - Compare to the Prepared pi0 Vectors

The prepared vector file contains `pi0_events`, with vectors such as `cluster_energy`, `cluster_row`, `cluster_column`, `pair_mass`, `pair_opening_angle_deg`, and `pair_separation_cm`.

Use these branches to check the scale of your result.

**Question 7:** Where is the strongest peak-like region in the prepared pi0 invariant-mass histogram? Is it exactly at the nominal pi0 mass, or shifted/broadened? What detector effects could cause that?


In [ ]:
def flatten_vector_branch(tree, branch_name, max_events=100000):
    out = []
    n = min(max_events, tree.GetEntries())
    for i in range(n):
        tree.GetEntry(i)
        vec = getattr(tree, branch_name)
        out.extend(float(x) for x in vec)
    return np.array(out, dtype=float)


def vector_lengths(tree, branch_name, max_events=100000):
    out = []
    n = min(max_events, tree.GetEntries())
    for i in range(n):
        tree.GetEntry(i)
        out.append(len(getattr(tree, branch_name)))
    return np.array(out, dtype=int)


In [ ]:
vec_f, pi0_tree = open_root_tree(VECTOR_FILE, "pi0_events")

# TODO: flatten pair_mass, pair_opening_angle_deg, and pair_separation_cm.
# Then make histograms and compare with the ROOT histograms below.

# masses = flatten_vector_branch(pi0_tree, "pair_mass", max_events=100000)
# plt.hist(masses, bins=120, range=(0, 400), histtype="step")
# plt.xlabel("two-cluster invariant mass [MeV]")
# plt.ylabel("candidate pairs")
# plt.show()

vec_f.Close()


In [ ]:
plot_root_hist(VECTOR_FILE, "pi0_invariant_mass_best_pair", xlabel="best-pair invariant mass [MeV]", logy=False)
plot_root_hist(VECTOR_FILE, "pi0_opening_angle_deg", xlabel="opening angle [deg]", logy=False)
plot_root_hist(VECTOR_FILE, "pi0_cluster_separation_cm", xlabel="cluster separation [cm]", logy=False)


## Final Questions

1. Which clustering threshold choices give stable-looking clusters?
2. How often do events have at least two clusters?
3. Why is cluster separation important for pi0 reconstruction?
4. Why can the reconstructed invariant-mass distribution be broad?
5. What would you check next if the pi0 peak is shifted from 135 MeV?
